In [ ]:
   pip list | findstr /i pydantic

In [ ]:
pip install --force-reinstall "pydantic<2.13"

## Setup

In [1]:
# Load libraries and the API key from .env
import base64
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
from anthropic.types import Message

# Create the client and choose the model
client = Anthropic()
model = "claude-haiku-4-5"

## Helper functions

In [2]:
# Add a user turn; accepts text, a list of blocks, or a Message
def add_user_message(messages, message):
    messages.append({
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    })

# Add an assistant turn
def add_assistant_message(messages, message):
    messages.append({
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    })

# Send the conversation to Claude and return the full message
def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1500,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    return client.messages.create(**params)

## Install a PDF library

In [ ]:
# fpdf2 lets us create a sample PDF, since the folder has none
%pip install fpdf2

## Create a sample PDF about Earth's atmosphere

In [3]:
from fpdf import FPDF

# One short text per page, so page numbers in citations are meaningful
pages = [
    "Early atmosphere. Earth formed about 4.5 billion years ago. Its first atmosphere "
    "was mostly hydrogen and helium, which escaped into space because Earth's gravity "
    "was too weak to hold them.",

    "Second atmosphere. Volcanic eruptions released water vapor, carbon dioxide, and "
    "nitrogen. As the planet cooled, the water vapor condensed and formed the oceans.",

    "Oxygen. About 2.4 billion years ago, cyanobacteria began producing oxygen through "
    "photosynthesis. This Great Oxidation Event slowly raised oxygen levels and "
    "eventually made complex life possible.",
]

# Build the PDF, adding each text on its own page
pdf = FPDF()
pdf.set_font("Helvetica", size=14)
for text in pages:
    pdf.add_page()
    pdf.multi_cell(0, 10, text)
pdf.output("earth.pdf")
print("Created earth.pdf with", len(pages), "pages")

Created earth.pdf with 3 pages


## Load the PDF as base64

In [4]:
# Read the PDF bytes and encode them as base64 text
with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

## Ask a question with citations enabled

In [5]:
messages = []
add_user_message(messages, [
    # Document block with the two new fields: title and citations
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes,
        },
        "title": "earth.pdf",
        "citations": {"enabled": True},
    },
    {"type": "text", "text": "How did Earth's atmosphere form, and when did oxygen appear?"},
])

response = chat(messages)

## Inspect the response blocks

In [6]:
# The answer is split into several text blocks, and some carry citations
for i, block in enumerate(response.content):
    print(f"Block {i}: {block.type}")
    print("  text:", block.text)
    for c in (getattr(block, "citations", None) or []):
        print("  citation ->", c.type)
        print("     cited_text:", c.cited_text)
        print("     document_title:", c.document_title)
        print("     pages:", c.start_page_number, "to", c.end_page_number)

Block 0: text
  text: # Formation of Earth's Atmosphere


Block 1: text
  text: Earth formed about 4.5 billion years ago with a first atmosphere mostly composed of hydrogen and helium, which escaped into space because Earth's gravity was too weak to hold them.
  citation -> page_location
     cited_text: Earth formed about 4.5 billion years ago. Its first atmosphere was
mostly hydrogen and helium, which escaped into space because Earth's gravity was
too weak to hold them.
     document_title: earth.pdf
     pages: 1 to 2
Block 2: text
  text: 


Block 3: text
  text: The second atmosphere formed when volcanic eruptions released water vapor, carbon dioxide, and nitrogen. As the planet cooled, the water vapor condensed and formed the oceans.
  citation -> page_location
     cited_text: Volcanic eruptions released water vapor, carbon dioxide, and
nitrogen. As the planet cooled, the water vapor condensed and formed the oceans.
     document_title: earth.pdf
     pages: 2 to 3
Block 4: text

Note that end_page_number is exclusive, so a citation from page 2 shows “2 to 3”.

## Show the answer with numbered references

In [7]:
# Build a readable answer where each citation becomes a marker like [1]
answer = ""
sources = []

for block in response.content:
    if block.type != "text":
        continue
    answer += block.text
    for c in (getattr(block, "citations", None) or []):
        sources.append(c)
        answer += f" [{len(sources)}]"  # marker after the cited text

print(answer)
print("\nSources:")
for n, c in enumerate(sources, start=1):
    print(f"[{n}] {c.document_title}, page {c.start_page_number}: \"{c.cited_text}\"")

# Formation of Earth's Atmosphere

Earth formed about 4.5 billion years ago with a first atmosphere mostly composed of hydrogen and helium, which escaped into space because Earth's gravity was too weak to hold them. [1]

The second atmosphere formed when volcanic eruptions released water vapor, carbon dioxide, and nitrogen. As the planet cooled, the water vapor condensed and formed the oceans. [2]

## Appearance of Oxygen

About 2.4 billion years ago, cyanobacteria began producing oxygen through photosynthesis. This Great Oxidation Event slowly raised oxygen levels and eventually made complex life possible. [3]

Sources:
[1] earth.pdf, page 1: "Earth formed about 4.5 billion years ago. Its first atmosphere was
mostly hydrogen and helium, which escaped into space because Earth's gravity was
too weak to hold them."
[2] earth.pdf, page 2: "Volcanic eruptions released water vapor, carbon dioxide, and
nitrogen. As the planet cooled, the water vapor condensed and formed the oceans."
[3] eart

## Citations with plain text

In [8]:
# The same idea with a plain text source instead of a PDF
article_text = "\n\n".join(pages)

messages = []
add_user_message(messages, [
    {
        "type": "document",
        "source": {
            "type": "text",
            "media_type": "text/plain",
            "data": article_text,
        },
        "title": "earth_article",
        "citations": {"enabled": True},
    },
    {"type": "text", "text": "What caused the oceans to form?"},
])

response = chat(messages)

# Text sources report character positions instead of page numbers
for block in response.content:
    print(block.text)
    for c in (getattr(block, "citations", None) or []):
        print("  cited:", c.cited_text)
        print("  chars:", c.start_char_index, "to", c.end_char_index)

Volcanic eruptions released water vapor, carbon dioxide, and nitrogen. As the planet cooled, the water vapor condensed and formed the oceans.
  cited: Volcanic eruptions released water vapor, carbon dioxide, and nitrogen. As the planet cooled, the water vapor condensed and formed the oceans.


  chars: 209 to 352
